# Pyramid AI: Training Memory Writer (Qwen 2.5 0.5B -> GGUF)
Этот блокнот обучает специализированную модель **Pyramid Memory Writer** на базе **Qwen 2.5 0.5B Instruct** и экспортирует её в `.gguf`.

**Требования:** бесплатная GPU (T4 в Colab). Время обучения: ~5-10 минут.

In [ ]:
# 1. Установка Unsloth и зависимостей
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
# 2. Загрузка базовой модели Qwen 2.5 0.5B Instruct в 4-бит
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
model_name = "unsloth/Qwen2.5-0.5B-Instruct"

print("Загрузка модели...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)

# Добавление LoRA адаптеров
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print("Модель готова к обучению!")

In [ ]:
# 3. Подготовка датасета (встроенный генератор)
import json
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are the Pyramid Memory Writer. Your sole responsibility is to extract "
    "salient facts, architectural decisions, knowledge, or session state into a strict "
    "JSON object matching the MemoryUpdate schema.\n\n"
    "Schema:\n"
    "{\n"
    "  \"key\": \"dot.separated.key\",\n"
    "  \"category\": \"project\" | \"knowledge\" | \"session\",\n"
    "  \"content\": \"...\",\n"
    "  \"tags\": [\"tag1\", \"tag2\"]\n"
    "}\n\n"
    "Output ONLY valid JSON. No conversational text, no markdown backticks."
)

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    # Скачаем сгенерированный датасет или встроим прямо сюда
    pass

# Если файла нет, генерируем базовые примеры прямо в блокноте
import urllib.request
examples = [
    {"input": "Ядро Pyramid написано на Python, а inference на llama.cpp.", "output": {"key": "project.tech_stack.engine", "category": "project", "content": "Pyramid engine in Python, inference via llama.cpp", "tags": ["python", "llama.cpp"]}},
    {"input": "Стадии оркестрации: Wide, Compress, Narrow, Verify.", "output": {"key": "project.architecture.stages", "category": "project", "content": "4 stages: Wide, Compress, Narrow, Verify", "tags": ["architecture", "stages"]}},
    {"input": "Формат хранения памяти решено сделать flat JSON с changelog.", "output": {"key": "project.storage.format", "category": "project", "content": "Flat JSON partitioned by category with changelog.json", "tags": ["memory", "json"]}},
    {"input": "В Python списки изменяемые, а кортежи неизменяемые.", "output": {"key": "knowledge.python.tuples_vs_lists", "category": "knowledge", "content": "Lists are mutable, tuples are immutable in Python", "tags": ["python", "types"]}},
    {"input": "Завершили этап Phase 2 и тестируем обучение собственной модели.", "output": {"key": "session.status.phase", "category": "session", "content": "Phase 2 completed, training custom model", "tags": ["status", "session"]}}
]

formatted_samples = []
for ex in examples * 30: # 150 сэмплов
    formatted_samples.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ex["input"]},
            {"role": "assistant", "content": json.dumps(ex["output"], ensure_ascii=False)}
        ]
    })

texts = [tokenizer.apply_chat_template(item["messages"], tokenize=False) for item in formatted_samples]
dataset = Dataset.from_dict({"text": texts})
print(f"Подготовлено {len(dataset)} обучающих примеров.")

In [ ]:
# 4. Обучение модели
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        warmup_steps=5,
        max_steps=50,
        learning_rate=3e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
    ),
)

print("Старт обучения...")
trainer.train()
print("Обучение завершено!")

In [ ]:
# 5. Тестирование обученной модели
FastLanguageModel.for_inference(model)
test_prompt = "Зафиксируй: мы используем SQLite для индексов графа зависимостей."
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=128, temperature=0.1)
decoded = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("Вход:", test_prompt)
print("Выход модели:", decoded)

In [ ]:
# 6. Экспорт в GGUF (Q4_K_M) и скачивание
print("Квантование и сохранение в GGUF...")
model.save_pretrained_gguf("pyramid_memory_writer", tokenizer, quantization_method="q4_k_m")

from google.colab import files
import glob
gguf_files = glob.glob("*Q4_K_M.gguf") + glob.glob("pyramid_memory_writer/*.gguf")
if gguf_files:
    print(f"Скачивание: {gguf_files[0]}")
    files.download(gguf_files[0])
else:
    print("Найденные файлы:", glob.glob("*.gguf"))